# Agent Memory: Building Memory-Enabled Investment Agents with LangGraph

In this notebook, we'll explore **agent memory systems** - the ability for AI agents to remember information across interactions. We'll implement all five memory types from the **CoALA (Cognitive Architectures for Language Agents)** framework while building a Stone Ridge Investment Advisory Assistant.

**Learning Objectives:**
- Understand the 5 memory types from the CoALA framework
- Implement short-term memory with checkpointers and thread_id
- Build long-term memory with InMemoryStore and namespaces
- Use semantic memory for meaning-based retrieval
- Apply episodic memory for few-shot learning from past experiences
- Create procedural memory for self-improving agents
- Combine all memory types into a unified investment advisory agent

## Table of Contents:

- **Breakout Room #1:** Memory Foundations
  - Task 1: Dependencies
  - Task 2: Understanding Agent Memory (CoALA Framework)
  - Task 3: Short-Term Memory (MemorySaver, thread_id)
  - Task 4: Long-Term Memory (InMemoryStore, namespaces)
  - Task 5: Message Trimming & Context Management
  - Question #1 & Question #2
  - 🏗️ Activity #1: Store & Retrieve User Investment Profile

- **Breakout Room #2:** Advanced Memory & Integration
  - Task 6: Semantic Memory (Embeddings + Search)
  - Task 7: Building Semantic Investment Knowledge Base
  - Task 8: Episodic Memory (Few-Shot Learning)
  - Task 9: Procedural Memory (Self-Improving Agent)
  - Task 10: Unified Investment Memory Agent
  - Question #3 & Question #4
  - 🏗️ Activity #2: Investment Memory Dashboard

---
# 🤝 Breakout Room #1
## Memory Foundations

## Task 1: Dependencies

Before we begin, make sure you have:

1. **API Keys** for:
   - OpenAI (for GPT-4o-mini and embeddings)
   - LangSmith (optional, for tracing)

2. **Dependencies installed** via `uv sync`

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

In [2]:
# Set API Keys
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

OpenAI API Key:  ········


In [3]:
# Optional: LangSmith for tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Agent Memory - Investment - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key (press Enter to skip): ") or ""

if not os.environ["LANGCHAIN_API_KEY"]:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")
else:
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")

LangSmith API Key (press Enter to skip):  ········


LangSmith tracing enabled. Project: AIE9 - Agent Memory - Investment - 5798d541


In [4]:
# Initialize LLM
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Test the connection
response = llm.invoke("Say 'Memory systems ready!' in exactly those words.")
print(response.content)

Memory systems ready!


## Task 2: Understanding Agent Memory (CoALA Framework)

The **CoALA (Cognitive Architectures for Language Agents)** framework identifies 5 types of memory that agents can use:

| Memory Type | Human Analogy | AI Implementation | Investment Example |
|-------------|---------------|-------------------|------------------|
| **Short-term** | What someone just said | Conversation history within a thread | Current investment consultation conversation |
| **Long-term** | Remembering a friend's birthday | User preferences stored across sessions | User's risk tolerance, portfolio size, investment goals |
| **Semantic** | Knowing Paris is in France | Facts retrieved by meaning | Investment knowledge retrieval |
| **Episodic** | Remembering your first day at work | Learning from past experiences | Past successful advisory patterns |
| **Procedural** | Knowing how to ride a bike | Self-improving instructions | Learned communication and advisory preferences |

### Memory Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                LangGraph Investment Advisory Agent               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐           │
│  │  Short-term  │  │  Long-term   │  │   Semantic   │           │
│  │    Memory    │  │    Memory    │  │    Memory    │           │
│  │              │  │              │  │              │           │
│  │ Checkpointer │  │    Store     │  │Store+Embed   │           │
│  │ + thread_id  │  │ + namespace  │  │  + search()  │           │
│  └──────────────┘  └──────────────┘  └──────────────┘           │
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐                             │
│  │   Episodic   │  │  Procedural  │                             │
│  │    Memory    │  │    Memory    │                             │
│  │              │  │              │                             │
│  │  Few-shot    │  │Self-modifying│                             │
│  │  examples    │  │   prompts    │                             │
│  └──────────────┘  └──────────────┘                             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Key LangGraph Components

| Component | Memory Type | Scope |
|-----------|-------------|-------|
| `MemorySaver` (Checkpointer) | Short-term | Within a single thread |
| `InMemoryStore` | Long-term, Semantic, Episodic, Procedural | Across all threads |
| `thread_id` | Short-term | Identifies unique conversations |
| Namespaces | All store-based | Organizes memories by user/purpose |

**Documentation:**
- [CoALA Paper](https://arxiv.org/abs/2309.02427)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)

## Task 3: Short-Term Memory (MemorySaver, thread_id)

**Short-term memory** maintains context within a single conversation thread. Think of it like your working memory during a phone call - you remember what was said earlier, but once the call ends, those details fade.

In LangGraph, short-term memory is implemented through:
- **Checkpointer**: Saves the graph state at each step
- **thread_id**: Uniquely identifies each conversation

### How It Works

```
Thread 1: "Hi, I'm Alice"          Thread 2: "What's my name?"
     │                                   │
     ▼                                   ▼
┌──────────────┐                   ┌──────────────┐
│ Checkpointer │                   │ Checkpointer │
│  thread_1    │                   │  thread_2    │
│              │                   │              │
│ ["Hi Alice"] │                   │ [empty]      │
└──────────────┘                   └──────────────┘
     │                                   │
     ▼                                   ▼
"Hi Alice!"                        "I don't know your name"
```

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Define the state schema for our graph
# The `add_messages` annotation tells LangGraph how to update the messages list
class State(TypedDict):
    messages: Annotated[list, add_messages]


# Define our investment chatbot node
def investment_chatbot(state: State):
    """Process the conversation and generate an investment-focused response."""
    system_prompt = SystemMessage(content="""You are a friendly Investment Advisory Assistant. 
Help users with questions about Stone Ridge's investment philosophy, market outlook, 
portfolio strategy, and risk management.
Be supportive and remember details the user shares about themselves.""")
    
    messages = [system_prompt] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build the graph
builder = StateGraph(State)
builder.add_node("chatbot", investment_chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Compile with a checkpointer for short-term memory
checkpointer = MemorySaver()
investment_graph = builder.compile(checkpointer=checkpointer)

print("Investment chatbot compiled with short-term memory (checkpointing)")

Investment chatbot compiled with short-term memory (checkpointing)


In [7]:
# Test short-term memory within a thread
config = {"configurable": {"thread_id": "investment_thread_1"}}

# First message - introduce ourselves
response = investment_graph.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alex and I want to understand Stone Ridge's investment approach.")]},
    config
)
print("User: Hi! My name is Alex and I want to understand Stone Ridge's investment approach.")
print(f"Assistant: {response['messages'][-1].content}")
print()

User: Hi! My name is Alex and I want to understand Stone Ridge's investment approach.
Assistant: Hi Alex! It's great to meet you. Stone Ridge has a unique investment philosophy that focuses on a few key principles. They emphasize a long-term, value-oriented approach, often looking for opportunities in areas that may be overlooked by traditional investors. 

Their investment strategy often includes a mix of alternative investments, such as real estate and private equity, alongside more traditional assets. They also utilize a systematic approach to risk management, aiming to protect capital while seeking attractive returns.

Is there a specific aspect of their investment approach that you're particularly interested in, or any specific goals you have in mind for your investments?



In [8]:
# Second message - test if it remembers (same thread)
response = investment_graph.invoke(
    {"messages": [HumanMessage(content="What's my name and what am I interested in learning about?")]},
    config  # Same config = same thread_id
)
print("User: What's my name and what am I interested in learning about?")
print(f"Assistant: {response['messages'][-1].content}")

User: What's my name and what am I interested in learning about?
Assistant: Your name is Alex, and you're interested in learning about Stone Ridge's investment approach. If there's anything specific you'd like to dive deeper into, such as their market outlook, portfolio strategy, or risk management practices, just let me know!


In [9]:
# New thread - it won't remember Alex!
different_config = {"configurable": {"thread_id": "investment_thread_2"}}

response = investment_graph.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    different_config  # Different thread_id = no memory of Alex
)
print("User (NEW thread): What's my name?")
print(f"Assistant: {response['messages'][-1].content}")
print()
print("Notice: The agent doesn't know our name because this is a new thread!")

User (NEW thread): What's my name?
Assistant: I'm sorry, but I don't have access to your name or any personal information unless you share it with me. How can I assist you today?

Notice: The agent doesn't know our name because this is a new thread!


In [10]:
# Inspect the state of thread 1
state = investment_graph.get_state(config)
print(f"Thread 1 has {len(state.values['messages'])} messages:")
for msg in state.values['messages']:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"  {role}: {content}")

Thread 1 has 4 messages:
  User: Hi! My name is Alex and I want to understand Stone Ridge's investment approach.
  Assistant: Hi Alex! It's great to meet you. Stone Ridge has a unique investment philosophy ...
  User: What's my name and what am I interested in learning about?
  Assistant: Your name is Alex, and you're interested in learning about Stone Ridge's investm...


## Task 4: Long-Term Memory (InMemoryStore, namespaces)

**Long-term memory** stores information across different conversation threads. This is like remembering that your friend prefers tea over coffee - you remember it every time you meet them, regardless of what you're currently discussing.

In LangGraph, long-term memory uses:
- **Store**: A persistent key-value store
- **Namespaces**: Organize memories by user, application, or context

### Key Difference from Short-Term Memory

| Short-Term (Checkpointer) | Long-Term (Store) |
|---------------------------|-------------------|
| Scoped to a single thread | Shared across all threads |
| Automatic (messages) | Explicit (you decide what to store) |
| Conversation history | User preferences, facts, etc. |

In [11]:
from langgraph.store.memory import InMemoryStore

# Create a store for long-term memory
store = InMemoryStore()

# Namespaces organize memories - typically by user_id and category
user_id = "user_alex"
profile_namespace = (user_id, "profile")
preferences_namespace = (user_id, "preferences")

# Store Alex's investment profile
store.put(profile_namespace, "name", {"value": "Alex"})
store.put(profile_namespace, "goals", {"primary": "long-term growth", "secondary": "income generation"})
store.put(profile_namespace, "constraints", {"risk_tolerance": "moderate", "restrictions": ["no tobacco stocks"], "esg_preference": True})
store.put(profile_namespace, "portfolio", {"size": "$500K", "horizon": "20 years", "current_allocation": "60/40 stocks/bonds"})

# Store Alex's preferences
store.put(preferences_namespace, "communication", {"style": "data-driven", "detail_level": "comprehensive"})
store.put(preferences_namespace, "reporting", {"frequency": "quarterly", "preferred_metrics": ["CAGR", "Sharpe ratio", "max drawdown"]})

print("Stored Alex's profile and preferences in long-term memory")

Stored Alex's profile and preferences in long-term memory


In [12]:
# Retrieve specific memories
name = store.get(profile_namespace, "name")
print(f"Name: {name.value}")

goals = store.get(profile_namespace, "goals")
print(f"Goals: {goals.value}")

# List all memories in a namespace
print("\nAll profile items:")
for item in store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Name: {'value': 'Alex'}
Goals: {'primary': 'long-term growth', 'secondary': 'income generation'}

All profile items:
  name: {'value': 'Alex'}
  goals: {'primary': 'long-term growth', 'secondary': 'income generation'}
  constraints: {'risk_tolerance': 'moderate', 'restrictions': ['no tobacco stocks'], 'esg_preference': True}
  portfolio: {'size': '$500K', 'horizon': '20 years', 'current_allocation': '60/40 stocks/bonds'}


In [13]:
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

# Define state with user_id for personalization
class PersonalizedState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def personalized_investment_chatbot(state: PersonalizedState, config: RunnableConfig, *, store: BaseStore):
    """An investment chatbot that uses long-term memory for personalization."""
    user_id = state["user_id"]
    profile_namespace = (user_id, "profile")
    preferences_namespace = (user_id, "preferences")
    
    # Retrieve user profile from long-term memory
    profile_items = list(store.search(profile_namespace))
    pref_items = list(store.search(preferences_namespace))
    
    # Build context from profile
    profile_text = "\n".join([f"- {p.key}: {p.value}" for p in profile_items])
    pref_text = "\n".join([f"- {p.key}: {p.value}" for p in pref_items])
    
    system_msg = f"""You are an Investment Advisory Assistant. You know the following about this user:

PROFILE:
{profile_text if profile_text else 'No profile stored.'}

PREFERENCES:
{pref_text if pref_text else 'No preferences stored.'}

Use this information to personalize your responses. Be supportive and helpful."""
    
    messages = [SystemMessage(content=system_msg)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build the personalized graph
builder2 = StateGraph(PersonalizedState)
builder2.add_node("chatbot", personalized_investment_chatbot)
builder2.add_edge(START, "chatbot")
builder2.add_edge("chatbot", END)

# Compile with BOTH checkpointer (short-term) AND store (long-term)
personalized_graph = builder2.compile(
    checkpointer=MemorySaver(),
    store=store
)

print("Personalized graph compiled with both short-term and long-term memory")

Personalized graph compiled with both short-term and long-term memory


In [14]:
# Test the personalized chatbot - it knows Alex's profile!
config = {"configurable": {"thread_id": "personalized_thread_1"}}

response = personalized_graph.invoke(
    {
        "messages": [HumanMessage(content="What investment strategy would you recommend for me?")],
        "user_id": "user_alex"
    },
    config
)

print("User: What investment strategy would you recommend for me?")
print(f"Assistant: {response['messages'][-1].content}")
print()
print("Notice: The agent knows about Alex's risk tolerance and portfolio without him mentioning it!")

User: What investment strategy would you recommend for me?
Assistant: Given your profile and investment goals, I recommend a diversified investment strategy that aligns with your focus on long-term growth and income generation while adhering to your moderate risk tolerance and ESG preferences. Here’s a comprehensive approach:

### 1. **Asset Allocation**
   - **Equities (60%)**: Focus on a mix of growth and dividend-paying stocks. Given your ESG preference, consider investing in:
     - **ESG-focused ETFs or mutual funds**: These funds typically invest in companies with strong environmental, social, and governance practices.
     - **Dividend Aristocrats**: Companies that have consistently increased their dividends over time, providing income and potential for capital appreciation.
   - **Fixed Income (40%)**: To balance your portfolio and provide income, consider:
     - **Corporate Bonds**: Look for investment-grade bonds from companies with strong ESG ratings.
     - **Municipal Bon

In [15]:
# Even in a NEW thread, it still knows Alex's profile
# because long-term memory is cross-thread!

new_config = {"configurable": {"thread_id": "personalized_thread_2"}}

response = personalized_graph.invoke(
    {
        "messages": [HumanMessage(content="Are there any risks I should be aware of given my portfolio?")],
        "user_id": "user_alex"
    },
    new_config
)

print("User (NEW thread): Are there any risks I should be aware of given my portfolio?")
print(f"Assistant: {response['messages'][-1].content}")
print()
print("Notice: Even in a new thread, the agent knows Alex's portfolio and constraints!")

User (NEW thread): Are there any risks I should be aware of given my portfolio?
Assistant: Given your portfolio's allocation of 60% stocks and 40% bonds, there are several risks to consider, especially in the context of your long-term growth and income generation goals, as well as your moderate risk tolerance:

1. **Market Risk**: With a significant portion of your portfolio in stocks, you are exposed to market volatility. Economic downturns can lead to declines in stock prices, which may affect your long-term growth potential.

2. **Interest Rate Risk**: Your bond allocation is subject to interest rate risk. If interest rates rise, the value of existing bonds may decline, which could impact your portfolio's overall performance, especially if you rely on bonds for income generation.

3. **Inflation Risk**: Over a 20-year horizon, inflation can erode the purchasing power of your returns. While stocks generally provide a hedge against inflation over the long term, it's important to ensur

## Task 5: Message Trimming & Context Management

Long conversations can exceed the LLM's context window. LangGraph provides utilities to manage message history:

- **`trim_messages`**: Keeps only recent messages up to a token limit
- **Summarization**: Compress older messages into summaries

### Why Trim Even with 128K Context?

Even with large context windows:
1. **Cost**: More tokens = higher API costs
2. **Latency**: Larger contexts take longer to process
3. **Quality**: Models can struggle with "lost in the middle" - important info buried in long contexts
4. **Relevance**: Old messages may not be relevant to current query

In [16]:
from langchain_core.messages import trim_messages

# Create a trimmer that keeps only recent messages
trimmer = trim_messages(
    max_tokens=500,  # Keep messages up to this token count
    strategy="last",  # Keep the most recent messages
    token_counter=llm,  # Use the LLM to count tokens
    include_system=True,  # Always keep system messages
    allow_partial=False,  # Don't cut messages in half
)

# Example: Create a long conversation
long_conversation = [
    SystemMessage(content="You are an investment advisory assistant."),
    HumanMessage(content="I want to improve my portfolio returns."),
    AIMessage(content="Great goal! Let's start with your current allocation. What does your portfolio look like?"),
    HumanMessage(content="I have about 60% stocks and 40% bonds."),
    AIMessage(content="That's a balanced allocation. For higher returns, you might consider increasing equity exposure or adding alternative investments."),
    HumanMessage(content="What about international diversification?"),
    AIMessage(content="International exposure can reduce risk through diversification. Consider allocating 20-30% to international developed and emerging markets."),
    HumanMessage(content="And alternative investments?"),
    AIMessage(content="Alternatives like reinsurance, real estate, and commodities can provide uncorrelated returns and enhance portfolio efficiency."),
    HumanMessage(content="What's the most important change I should make first?"),
]

# Trim to fit context window
trimmed = trimmer.invoke(long_conversation)
print(f"Original: {len(long_conversation)} messages")
print(f"Trimmed: {len(trimmed)} messages")
print("\nTrimmed conversation:")
for msg in trimmed:
    role = type(msg).__name__.replace("Message", "")
    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"  {role}: {content}")

Original: 10 messages
Trimmed: 10 messages

Trimmed conversation:
  System: You are an investment advisory assistant.
  Human: I want to improve my portfolio returns.
  AI: Great goal! Let's start with your current allocation. What d...
  Human: I have about 60% stocks and 40% bonds.
  AI: That's a balanced allocation. For higher returns, you might ...
  Human: What about international diversification?
  AI: International exposure can reduce risk through diversificati...
  Human: And alternative investments?
  AI: Alternatives like reinsurance, real estate, and commodities ...
  Human: What's the most important change I should make first?


In [17]:
# Summarization approach for longer conversations

def summarize_conversation(messages: list, max_messages: int = 6) -> list:
    """Summarize older messages to manage context length."""
    if len(messages) <= max_messages:
        return messages
    
    # Keep the system message and last few messages
    system_msg = messages[0] if isinstance(messages[0], SystemMessage) else None
    content_messages = messages[1:] if system_msg else messages
    
    if len(content_messages) <= max_messages:
        return messages
    
    old_messages = content_messages[:-max_messages+1]
    recent_messages = content_messages[-max_messages+1:]
    
    # Summarize old messages
    summary_prompt = f"""Summarize this conversation in 2-3 sentences, 
capturing key investment topics discussed and any important user information:

{chr(10).join([f'{type(m).__name__}: {m.content[:200]}' for m in old_messages])}"""
    
    summary = llm.invoke(summary_prompt)
    
    # Return: system + summary + recent messages
    result = []
    if system_msg:
        result.append(system_msg)
    result.append(SystemMessage(content=f"[Previous conversation summary: {summary.content}]"))
    result.extend(recent_messages)
    
    return result


# Test summarization
summarized = summarize_conversation(long_conversation, max_messages=4)
print(f"Summarized: {len(summarized)} messages")
print("\nSummarized conversation:")
for msg in summarized:
    role = type(msg).__name__.replace("Message", "")
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"  {role}: {content}")

Summarized: 5 messages

Summarized conversation:
  System: You are an investment advisory assistant.
  System: [Previous conversation summary: The user aims to improve their portfolio returns...
  Human: And alternative investments?
  AI: Alternatives like reinsurance, real estate, and commodities can provide uncorrel...
  Human: What's the most important change I should make first?


---
## ❓ Question #1:

What are the trade-offs between **short-term memory** (checkpointer) vs **long-term memory** (store)? When should investment data move from short-term to long-term? Consider:
- What information should persist across sessions?
- What are the compliance implications?
- How would you decide what to promote from short-term to long-term?

##### Answer:

Short-term memory is easier to configure, and allows an easy way to interact with an agent with or without including memory, which could be convenient for "fresh session" type queries. It has the obvious downside of only being available in the current context.

Long-term memory has longer and more robust persistence and can be used across multiple interactions, creating a better user experience to some extent. It also has a more structured format, allowing you to add items as needed.

There could be compliance or even just unexpected user experience depending on which one is used. People might be surprised that "reloading" a session either keeps or doesn't keep memory, depending on what was used.

One way to decide when to promote would be to inspect what short-term memory is being used frequently, and then over time persist that to a long-term store.

## ❓ Question #2:

Why use message trimming with a 128K context window when the Stone Ridge investor letter is relatively small? What should **always** be preserved when trimming an investment consultation?

Consider:
- The "lost in the middle" phenomenon
- Cost and latency implications
- What user information is critical for safety (risk tolerance, constraints, etc.)

##### Answer:

Some considerations are

* Cost – larger contexts will result in more tokens, increasing time and cost.
* Quality – LLMs tend to focus on the beginning and ends of the context, resulting in information loss from the middle.
* Importance – not all information is equal in the context. For an investment consultant, things like risk tolerance or the amount of assets being managed are important to keep in context.

---
## 🏗️ Activity #1: Store & Retrieve User Investment Profile

Build a complete investment profile system that:
1. Defines an investment profile schema (name, risk tolerance, portfolio size, investment horizon, restrictions, goals)
2. Creates functions to store and retrieve profile data
3. Builds a personalized investment agent that uses the profile
4. Tests that different users get different advice

### Requirements:
- Define at least 5 profile attributes
- Support multiple users with different profiles
- Agent should reference profile data in responses

In [19]:
### YOUR CODE HERE ###

# Step 1: Define an investment profile schema
# Example attributes: name, risk_tolerance, portfolio_size, investment_horizon, restrictions, goals, preferred_asset_classes

class Profile(TypedDict):
    name: str
    risk_tolerance: str  # "conservative", "moderate", "aggressive"
    portfolio_size: str
    investment_horizon: str
    restrictions: list[str]
    goals: list[str]
    preferred_asset_classes: list[str]


# Step 2: Create helper functions to store and retrieve profiles
def store_investment_profile(store, user_id: str, profile: dict):
    """Store a user's investment profile."""
    profile_namespace = (user_id, "profile")
    
    # Store each profile attribute as a separate key for easier retrieval
    for key, value in profile.items():
        store.put(profile_namespace, key, {"value": value})
    
    print(f"Stored profile for {user_id}")


def get_investment_profile(store, user_id: str) -> dict:
    """Retrieve a user's investment profile."""
    profile_namespace = (user_id, "profile")
    
    # Retrieve all profile items
    items = list(store.search(profile_namespace))
    
    if not items:
        return {}
    
    # Reconstruct profile dictionary
    profile = {}
    for item in items:
        profile[item.key] = item.value["value"]
    
    return profile


# Step 3: Create two different user profiles

# Create an InMemoryStore for profiles
profile_store = InMemoryStore()

# Profile 1: Conservative investor near retirement
profile_sarah = {
    "name": "Sarah Chen",
    "risk_tolerance": "conservative",
    "portfolio_size": "$2.5M",
    "investment_horizon": "5 years until retirement",
    "restrictions": ["no speculative investments", "avoid high-volatility assets"],
    "goals": ["capital preservation", "steady income generation", "maintain purchasing power"],
    "preferred_asset_classes": ["investment-grade bonds", "dividend aristocrats", "REITs"]
}

store_investment_profile(profile_store, "user_sarah", profile_sarah)

# Profile 2: Aggressive young investor
profile_marcus = {
    "name": "Marcus Rodriguez",
    "risk_tolerance": "aggressive",
    "portfolio_size": "$150K",
    "investment_horizon": "30+ years",
    "restrictions": ["ESG-focused only", "no fossil fuels"],
    "goals": ["maximum long-term growth", "wealth accumulation", "early retirement"],
    "preferred_asset_classes": ["growth stocks", "emerging markets", "venture capital", "crypto"]
}

store_investment_profile(profile_store, "user_marcus", profile_marcus)


# Step 4: Build a personalized agent that uses profiles

class ProfileBasedState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def profile_aware_chatbot(state: ProfileBasedState, config: RunnableConfig, *, store: BaseStore):
    """Investment chatbot that personalizes advice based on user profile."""
    user_id = state["user_id"]
    
    # Retrieve user profile
    profile = get_investment_profile(store, user_id)
    
    if not profile:
        system_msg = "You are an Investment Advisory Assistant. Ask the user about their investment profile first."
    else:
        # Build detailed context from profile
        profile_summary = f"""User Profile:
- Name: {profile.get('name', 'Unknown')}
- Risk Tolerance: {profile.get('risk_tolerance', 'Unknown')}
- Portfolio Size: {profile.get('portfolio_size', 'Unknown')}
- Investment Horizon: {profile.get('investment_horizon', 'Unknown')}
- Restrictions: {', '.join(profile.get('restrictions', []))}
- Goals: {', '.join(profile.get('goals', []))}
- Preferred Asset Classes: {', '.join(profile.get('preferred_asset_classes', []))}"""
        
        system_msg = f"""You are an Investment Advisory Assistant for Stone Ridge. 

{profile_summary}

Provide personalized investment advice that:
1. Matches their risk tolerance level
2. Aligns with their investment goals
3. Respects their restrictions
4. Considers their time horizon
5. Leverages their preferred asset classes

Be specific and actionable in your recommendations."""
    
    messages = [SystemMessage(content=system_msg)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build the profile-aware graph
profile_builder = StateGraph(ProfileBasedState)
profile_builder.add_node("chatbot", profile_aware_chatbot)
profile_builder.add_edge(START, "chatbot")
profile_builder.add_edge("chatbot", END)

profile_graph = profile_builder.compile(
    checkpointer=MemorySaver(),
    store=profile_store
)

print("Profile-aware investment assistant ready!")


# Step 5: Test with different users - they should get different advice

print("\n" + "="*80)
print("TESTING WITH SARAH (Conservative, near retirement)")
print("="*80)

config_sarah = {"configurable": {"thread_id": "sarah_thread_1"}}

response_sarah = profile_graph.invoke(
    {
        "messages": [HumanMessage(content="What investment strategy would you recommend for me?")],
        "user_id": "user_sarah"
    },
    config_sarah
)

print(f"\nSarah asks: What investment strategy would you recommend for me?")
print(f"\nAssistant's response for Sarah:")
print(response_sarah['messages'][-1].content)

print("\n" + "="*80)
print("TESTING WITH MARCUS (Aggressive, young investor)")
print("="*80)

config_marcus = {"configurable": {"thread_id": "marcus_thread_1"}}

response_marcus = profile_graph.invoke(
    {
        "messages": [HumanMessage(content="What investment strategy would you recommend for me?")],
        "user_id": "user_marcus"
    },
    config_marcus
)

print(f"\nMarcus asks: What investment strategy would you recommend for me?")
print(f"\nAssistant's response for Marcus:")
print(response_marcus['messages'][-1].content)

print("\n" + "="*80)
print("COMPARISON:")
print("="*80)
print("Notice how the advice differs based on:")
print("  • Sarah gets conservative, income-focused recommendations")
print("  • Marcus gets aggressive, growth-oriented recommendations")
print("  • Both respect the user's specific restrictions and goals")

Stored profile for user_sarah
Stored profile for user_marcus
Profile-aware investment assistant ready!

TESTING WITH SARAH (Conservative, near retirement)

Sarah asks: What investment strategy would you recommend for me?

Assistant's response for Sarah:
Given your conservative risk tolerance, investment horizon of 5 years until retirement, and your goals of capital preservation, steady income generation, and maintaining purchasing power, I recommend a diversified investment strategy that focuses on stability and income. Here’s a specific and actionable plan tailored to your profile:

### 1. **Investment-Grade Bonds (40% of Portfolio)**
   - **Allocation**: $1,000,000
   - **Strategy**: Invest in a mix of U.S. Treasury bonds, municipal bonds, and high-quality corporate bonds. Consider bond funds or ETFs that focus on investment-grade securities to provide diversification and liquidity.
   - **Example Funds**: 
     - Vanguard Intermediate-Term Investment-Grade Fund (VFICX)
     - iShare

---
# 🤝 Breakout Room #2
## Advanced Memory & Integration

## Task 6: Semantic Memory (Embeddings + Search)

**Semantic memory** stores facts and retrieves them based on *meaning* rather than exact matches. This is like how you might remember "that fund with the great risk-adjusted returns" even if you can't remember its exact name.

In LangGraph, semantic memory uses:
- **Store with embeddings**: Converts text to vectors for similarity search
- **`store.search()`**: Finds relevant memories by semantic similarity

### How It Works

```
User asks: "What helps with portfolio diversification?"
         ↓
Query embedded → [0.2, 0.8, 0.1, ...]
         ↓
Compare with stored investment facts:
  - "Uncorrelated assets reduce portfolio risk" → 0.92 similarity ✓
  - "Rebalancing maintains target allocations" → 0.35 similarity
         ↓
Return: "Uncorrelated assets reduce portfolio risk"
```

In [20]:
from langchain_openai import OpenAIEmbeddings

# Create embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a store with semantic search enabled
semantic_store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,  # Dimension of text-embedding-3-small
    }
)

print("Semantic memory store created with embedding support")

Semantic memory store created with embedding support


In [21]:
# Store various investment facts as semantic memories
namespace = ("investment", "facts")

investment_facts = [
    ("fact_1", {"text": "Diversification across uncorrelated assets can reduce portfolio risk without sacrificing returns"}),
    ("fact_2", {"text": "Stone Ridge focuses on alternative risk premiums including reinsurance and longevity risk"}),
    ("fact_3", {"text": "Tail risk hedging provides insurance against extreme market downturns"}),
    ("fact_4", {"text": "A long-term investment horizon allows investors to capture illiquidity premiums"}),
    ("fact_5", {"text": "Factor investing targets specific drivers of return such as value, momentum, and quality"}),
    ("fact_6", {"text": "Rebalancing portfolios periodically helps maintain target risk levels"}),
    ("fact_7", {"text": "Alternative investments like reinsurance have low correlation with traditional stock and bond markets"}),
    ("fact_8", {"text": "Systematic risk management frameworks help identify and mitigate portfolio vulnerabilities"}),
]

for key, value in investment_facts:
    semantic_store.put(namespace, key, value)

print(f"Stored {len(investment_facts)} investment facts in semantic memory")

Stored 8 investment facts in semantic memory


In [22]:
# Search semantically - notice we don't need exact matches!

queries = [
    "How can I protect my portfolio from a market crash?",
    "What alternative investments should I consider?",
    "How should I think about risk in my portfolio?",
    "What is Stone Ridge's investment approach?",
]

for query in queries:
    print(f"\nQuery: {query}")
    results = semantic_store.search(namespace, query=query, limit=2)
    for r in results:
        print(f"   {r.value['text']} (score: {r.score:.3f})")


Query: How can I protect my portfolio from a market crash?
   Tail risk hedging provides insurance against extreme market downturns (score: 0.454)
   Rebalancing portfolios periodically helps maintain target risk levels (score: 0.442)

Query: What alternative investments should I consider?
   Alternative investments like reinsurance have low correlation with traditional stock and bond markets (score: 0.568)
   Diversification across uncorrelated assets can reduce portfolio risk without sacrificing returns (score: 0.407)

Query: How should I think about risk in my portfolio?
   Rebalancing portfolios periodically helps maintain target risk levels (score: 0.492)
   Systematic risk management frameworks help identify and mitigate portfolio vulnerabilities (score: 0.471)

Query: What is Stone Ridge's investment approach?
   Stone Ridge focuses on alternative risk premiums including reinsurance and longevity risk (score: 0.641)
   Factor investing targets specific drivers of return such as

## Task 7: Building Semantic Investment Knowledge Base

Let's load the Stone Ridge 2025 Investor Letter and create a semantic knowledge base that our agent can search.

This is similar to RAG from Module 4, but now using LangGraph's Store API instead of a separate vector database.

In [23]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load and chunk the investment document
loader = PyMuPDFLoader("data/Stone Ridge 2025 Investor Letter.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documents)

print(f"Loaded and split into {len(chunks)} chunks")
print(f"\nSample chunk:\n{chunks[0].page_content[:200]}...")

Loaded and split into 127 chunks

Sample chunk:
2025 Investor Letter...


In [24]:
# Store chunks in semantic memory
knowledge_namespace = ("investment", "knowledge")

for i, chunk in enumerate(chunks):
    semantic_store.put(
        knowledge_namespace,
        f"chunk_{i}",
        {"text": chunk.page_content, "source": "Stone Ridge 2025 Investor Letter.pdf"}
    )

print(f"Stored {len(chunks)} chunks in semantic knowledge base")

Stored 127 chunks in semantic knowledge base


In [25]:
# Build a semantic search investment chatbot

class SemanticState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def semantic_investment_chatbot(state: SemanticState, config: RunnableConfig, *, store: BaseStore):
    """An investment chatbot that retrieves relevant facts using semantic search."""
    user_message = state["messages"][-1].content
    
    # Search for relevant knowledge
    knowledge_results = store.search(
        ("investment", "knowledge"),
        query=user_message,
        limit=3
    )
    
    # Build context from retrieved knowledge
    if knowledge_results:
        knowledge_text = "\n\n".join([f"- {r.value['text']}" for r in knowledge_results])
        system_msg = f"""You are an Investment Advisory Assistant with access to the Stone Ridge investor letter knowledge base.

Relevant information from your knowledge base:
{knowledge_text}

Use this information to answer the user's question. If the information doesn't directly answer their question, use your general knowledge but mention what you found."""
    else:
        system_msg = "You are an Investment Advisory Assistant. Answer investment questions helpfully."
    
    messages = [SystemMessage(content=system_msg)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build and compile
builder3 = StateGraph(SemanticState)
builder3.add_node("chatbot", semantic_investment_chatbot)
builder3.add_edge(START, "chatbot")
builder3.add_edge("chatbot", END)

semantic_graph = builder3.compile(
    checkpointer=MemorySaver(),
    store=semantic_store
)

print("Semantic investment chatbot ready")

Semantic investment chatbot ready


In [26]:
# Test semantic retrieval
config = {"configurable": {"thread_id": "semantic_thread_1"}}

questions = [
    "What is Stone Ridge's view on the current market environment?",
    "How does Stone Ridge approach risk management?",
    "What is Stone Ridge's investment philosophy?",
]

for q in questions:
    response = semantic_graph.invoke(
        {"messages": [HumanMessage(content=q)], "user_id": "test_user"},
        config
    )
    print(f"\nUser: {q}")
    print(f"Assistant: {response['messages'][-1].content[:500]}...")


User: What is Stone Ridge's view on the current market environment?
Assistant: The information I have access to does not provide specific insights or views from Stone Ridge regarding the current market environment. It emphasizes that any opinions or analyses shared do not represent a formal or official view of Stone Ridge and should not be interpreted as investment recommendations. 

For the most accurate and up-to-date perspective on Stone Ridge's views, I recommend checking their latest investor letters or official communications directly from the firm....

User: How does Stone Ridge approach risk management?
Assistant: The information I have does not provide specific details on Stone Ridge's approach to risk management. However, it does highlight that their operating model has been characterized by a focus on building products they want for themselves, which suggests a careful consideration of risk in their product design and investment strategies.

In general, effective risk manag

## Task 8: Episodic Memory (Few-Shot Learning)

**Episodic memory** stores past experiences and uses them as examples for future tasks. In humans, this is like remembering the first time you provided a successful investment recommendation and using that experience to guide future interactions.

For AI agents, episodic memory is often implemented as **few-shot examples** - showing the model examples of correct behavior so it can learn patterns.

### The CoALA Framework Perspective

> "Facts can be written to semantic memory, whereas **experiences** can be written to episodic memory."

Key insight: Sometimes it's easier to **show** than **tell**. LLMs learn well from examples!

In [27]:
# Create episodic memory with past successful interactions
episodic_namespace = ("agent", "episodes")

successful_episodes = [
    {
        "situation": "User asked about portfolio diversification with concentrated holdings",
        "input": "I have most of my portfolio in tech stocks. How should I diversify?",
        "output": """For a concentrated tech portfolio, I'd recommend diversifying across several dimensions:

1. **Asset classes** - Add bonds, real estate (REITs), and alternatives
2. **Geographies** - Include international developed and emerging markets
3. **Sectors** - Shift allocation toward healthcare, consumer staples, and industrials
4. **Alternative risk premiums** - Consider reinsurance or longevity-linked investments

Start by gradually rebalancing 5-10% per quarter to avoid tax impacts. Would you like a specific rebalancing plan?""",
        "feedback": "User appreciated the structured approach and specific action items"
    },
    {
        "situation": "User concerned about market downturn impact on retirement portfolio",
        "input": "I'm 10 years from retirement and worried about a market crash. What should I do?",
        "output": """Market timing is difficult, but there are proven strategies to protect near-retirement portfolios:

**Risk reduction:**
- Gradually shift to a more conservative allocation (e.g., 50/50 stocks/bonds)
- Consider tail risk hedging for catastrophic downside protection

**Income stability:**
- Build a 2-3 year cash buffer for living expenses
- Add dividend-paying stocks and investment-grade bonds

**Stay invested:**
- Keep some equity exposure for growth to combat inflation
- Focus on quality companies with strong balance sheets

Which aspect would you like to explore in more detail?""",
        "feedback": "User found the balanced approach reassuring and actionable"
    },
    {
        "situation": "User asking about alternative investments",
        "input": "What are alternative investments and should I have them in my portfolio?",
        "output": """Alternative investments are assets beyond traditional stocks and bonds. They can add diversification and unique return sources:

**Common alternatives:**
- Real estate (REITs, direct property)
- Private equity and venture capital
- Hedge funds and managed futures
- Reinsurance and catastrophe bonds
- Commodities and natural resources

**Benefits:**
- Low correlation with traditional markets
- Access to different risk premiums
- Potential for higher risk-adjusted returns

**Considerations:**
- Often less liquid than public markets
- May require higher minimum investments
- Due diligence is critical

A typical allocation might be 10-20% of your portfolio. What's your current portfolio size and investment horizon?""",
        "feedback": "User valued the comprehensive overview without being overwhelming"
    },
]

for i, episode in enumerate(successful_episodes):
    semantic_store.put(
        episodic_namespace,
        f"episode_{i}",
        {
            "text": episode["situation"],  # Used for semantic search
            **episode
        }
    )

print(f"Stored {len(successful_episodes)} episodic memories (past successful interactions)")

Stored 3 episodic memories (past successful interactions)


In [28]:
class EpisodicState(TypedDict):
    messages: Annotated[list, add_messages]


def episodic_investment_chatbot(state: EpisodicState, config: RunnableConfig, *, store: BaseStore):
    """A chatbot that learns from past successful interactions."""
    user_question = state["messages"][-1].content
    
    # Search for similar past experiences
    similar_episodes = store.search(
        ("agent", "episodes"),
        query=user_question,
        limit=1
    )
    
    # Build few-shot examples from past episodes
    if similar_episodes:
        episode = similar_episodes[0].value
        few_shot_example = f"""Here's an example of a similar investment question I handled well:

User asked: {episode['input']}

My response was:
{episode['output']}

The user feedback was: {episode['feedback']}

Use this as inspiration for the style, structure, and tone of your response, but tailor it to the current question."""
        
        system_msg = f"""You are an Investment Advisory Assistant. Learn from your past successes:

{few_shot_example}"""
    else:
        system_msg = "You are an Investment Advisory Assistant. Be helpful, specific, and supportive."
    
    messages = [SystemMessage(content=system_msg)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build the episodic memory graph
builder4 = StateGraph(EpisodicState)
builder4.add_node("chatbot", episodic_investment_chatbot)
builder4.add_edge(START, "chatbot")
builder4.add_edge("chatbot", END)

episodic_graph = builder4.compile(
    checkpointer=MemorySaver(),
    store=semantic_store
)

print("Episodic memory chatbot ready")

Episodic memory chatbot ready


In [29]:
# Test episodic memory - similar question to stored episode
config = {"configurable": {"thread_id": "episodic_thread_1"}}

response = episodic_graph.invoke(
    {"messages": [HumanMessage(content="I'm thinking about adding some alternative investments to my portfolio. What should I consider?")]},
    config
)

print("User: I'm thinking about adding some alternative investments to my portfolio. What should I consider?")
print(f"\nAssistant: {response['messages'][-1].content}")
print("\nNotice: The response structure mirrors the successful alternatives episode!")

User: I'm thinking about adding some alternative investments to my portfolio. What should I consider?

Assistant: Adding alternative investments to your portfolio can be a strategic move to enhance diversification and potentially improve returns. Here are some key considerations to keep in mind:

**Types of Alternative Investments:**
- **Real Estate:** This includes Real Estate Investment Trusts (REITs) or direct property ownership, which can provide rental income and capital appreciation.
- **Private Equity:** Investing in private companies or funds that target private equity can offer high returns, though they often come with higher risk and longer investment horizons.
- **Hedge Funds:** These funds employ various strategies to generate returns, including long/short equity, market neutral, and global macro strategies.
- **Commodities:** Investing in physical goods like gold, oil, or agricultural products can hedge against inflation and provide diversification.
- **Cryptocurrencies:**

## Task 9: Procedural Memory (Self-Improving Agent)

**Procedural memory** stores the rules and instructions that guide behavior. In humans, this is like knowing *how* to give good advice - it's internalized knowledge about performing tasks.

For AI agents, procedural memory often means **self-modifying prompts**. The agent can:
1. Store its current instructions in the memory store
2. Reflect on feedback from interactions
3. Update its own instructions to improve

### The Reflection Pattern

```
User feedback: "Your advice is too long and complicated"
         ↓
Agent reflects on current instructions
         ↓
Agent updates instructions: "Keep advice concise and actionable"
         ↓
Future responses use updated instructions
```

In [30]:
# Initialize procedural memory with base instructions
procedural_namespace = ("agent", "instructions")

initial_instructions = """You are an Investment Advisory Assistant.

Guidelines:
- Be objective and data-driven in your analysis
- Provide evidence-based investment information
- Ask clarifying questions about risk tolerance and investment goals
- Present balanced perspectives on investment decisions
- Always note that past performance doesn't guarantee future results"""

semantic_store.put(
    procedural_namespace,
    "investment_assistant",
    {"instructions": initial_instructions, "version": 1}
)

print("Initialized procedural memory with base instructions")
print(f"\nCurrent Instructions (v1):\n{initial_instructions}")

Initialized procedural memory with base instructions

Current Instructions (v1):
You are an Investment Advisory Assistant.

Guidelines:
- Be objective and data-driven in your analysis
- Provide evidence-based investment information
- Ask clarifying questions about risk tolerance and investment goals
- Present balanced perspectives on investment decisions
- Always note that past performance doesn't guarantee future results


In [31]:
class ProceduralState(TypedDict):
    messages: Annotated[list, add_messages]
    feedback: str  # Optional feedback from user


def get_instructions(store: BaseStore) -> tuple[str, int]:
    """Retrieve current instructions from procedural memory."""
    item = store.get(("agent", "instructions"), "investment_assistant")
    if item is None:
        return "You are a helpful investment advisory assistant.", 0
    return item.value["instructions"], item.value["version"]


def procedural_assistant_node(state: ProceduralState, config: RunnableConfig, *, store: BaseStore):
    """Respond using current procedural instructions."""
    instructions, version = get_instructions(store)
    
    messages = [SystemMessage(content=instructions)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


def reflection_node(state: ProceduralState, config: RunnableConfig, *, store: BaseStore):
    """Reflect on feedback and update instructions if needed."""
    feedback = state.get("feedback", "")
    
    if not feedback:
        return {}  # No feedback, no update needed
    
    # Get current instructions
    current_instructions, version = get_instructions(store)
    
    # Ask the LLM to reflect and improve instructions
    reflection_prompt = f"""You are improving an investment advisory assistant's instructions based on user feedback.

Current Instructions:
{current_instructions}

User Feedback:
{feedback}

Based on this feedback, provide improved instructions. Keep the same general format but incorporate the feedback.
Only output the new instructions, nothing else."""
    
    response = llm.invoke([HumanMessage(content=reflection_prompt)])
    new_instructions = response.content
    
    # Update procedural memory with new instructions
    store.put(
        ("agent", "instructions"),
        "investment_assistant",
        {"instructions": new_instructions, "version": version + 1}
    )
    
    print(f"\nInstructions updated to version {version + 1}")
    return {}


def should_reflect(state: ProceduralState) -> str:
    """Decide whether to reflect on feedback."""
    if state.get("feedback"):
        return "reflect"
    return "end"


# Build the procedural memory graph
builder5 = StateGraph(ProceduralState)
builder5.add_node("assistant", procedural_assistant_node)
builder5.add_node("reflect", reflection_node)

builder5.add_edge(START, "assistant")
builder5.add_conditional_edges("assistant", should_reflect, {"reflect": "reflect", "end": END})
builder5.add_edge("reflect", END)

procedural_graph = builder5.compile(
    checkpointer=MemorySaver(),
    store=semantic_store
)

print("Procedural memory graph ready (with self-improvement capability)")

Procedural memory graph ready (with self-improvement capability)


In [32]:
# Test with initial instructions
config = {"configurable": {"thread_id": "procedural_thread_1"}}

response = procedural_graph.invoke(
    {
        "messages": [HumanMessage(content="How should I think about portfolio risk?")],
        "feedback": ""  # No feedback yet
    },
    config
)

print("User: How should I think about portfolio risk?")
print(f"\nAssistant (v1 instructions):\n{response['messages'][-1].content}")

User: How should I think about portfolio risk?

Assistant (v1 instructions):
Thinking about portfolio risk involves understanding several key concepts and factors that can influence the performance of your investments. Here are some important considerations:

1. **Risk Tolerance**: Assess your own risk tolerance, which is your ability and willingness to endure fluctuations in the value of your investments. This can be influenced by factors such as your investment goals, time horizon, and financial situation. Would you consider yourself conservative, moderate, or aggressive in your investment approach?

2. **Diversification**: Diversifying your portfolio across different asset classes (stocks, bonds, real estate, etc.) and sectors can help mitigate risk. By spreading investments, you reduce the impact of a poor-performing asset on your overall portfolio. Have you considered how diversified your current portfolio is?

3. **Volatility**: Understand the volatility of the assets in your por

In [33]:
# Now provide feedback - the agent will update its own instructions!
response = procedural_graph.invoke(
    {
        "messages": [HumanMessage(content="How should I think about portfolio risk?")],
        "feedback": "Your responses are too long. Please be more concise and give me 3 actionable insights maximum."
    },
    {"configurable": {"thread_id": "procedural_thread_2"}}
)


Instructions updated to version 2


In [34]:
# Check the updated instructions
new_instructions, version = get_instructions(semantic_store)
print(f"Updated Instructions (v{version}):\n")
print(new_instructions)

Updated Instructions (v2):

You are an Investment Advisory Assistant.

Guidelines:
- Be objective and data-driven in your analysis.
- Provide up to 3 actionable insights based on evidence-based investment information.
- Ask clarifying questions about risk tolerance and investment goals.
- Present balanced perspectives on investment decisions.
- Always note that past performance doesn't guarantee future results.


In [35]:
# Test with updated instructions - should be more concise now!
response = procedural_graph.invoke(
    {
        "messages": [HumanMessage(content="What investment opportunities should I consider in the current market?")],
        "feedback": ""  # No feedback this time
    },
    {"configurable": {"thread_id": "procedural_thread_3"}}
)

print(f"User: What investment opportunities should I consider in the current market?")
print(f"\nAssistant (v{version} instructions - after feedback):")
print(response['messages'][-1].content)
print("\nNotice: The response should now be more concise based on the feedback!")

User: What investment opportunities should I consider in the current market?

Assistant (v2 instructions - after feedback):
To provide tailored investment opportunities, I would need to understand your risk tolerance and investment goals. Here are a few clarifying questions:

1. What is your risk tolerance? Are you more conservative, moderate, or aggressive in your investment approach?
2. What are your investment goals? Are you looking for long-term growth, income generation, or capital preservation?
3. What is your investment horizon? Are you looking to invest for the short term, medium term, or long term?

In the current market, here are three actionable insights based on recent trends and data:

1. **Diversified ETFs**: Exchange-Traded Funds (ETFs) that focus on sectors like technology, healthcare, or renewable energy have shown resilience and growth potential. Consider ETFs that track indices with strong fundamentals and growth prospects. However, ensure that the sectors align with

## Task 10: Unified Investment Memory Agent

Now let's combine **all 5 memory types** into a unified investment advisory agent:

1. **Short-term**: Remembers current conversation (checkpointer)
2. **Long-term**: Stores user profile across sessions (store + namespace)
3. **Semantic**: Retrieves relevant investment knowledge (store + embeddings)
4. **Episodic**: Uses past successful interactions as examples (store + search)
5. **Procedural**: Adapts behavior based on feedback (store + reflection)

### Memory Retrieval Flow

```
User Query: "What investment strategy suits my risk profile?"
              │
              ▼
┌─────────────────────────────────────────────────┐
│  1. PROCEDURAL: Get current instructions         │
│  2. LONG-TERM: Load user profile (constraints)   │
│  3. SEMANTIC: Search investment knowledge        │
│  4. EPISODIC: Find similar past interactions     │
│  5. SHORT-TERM: Include conversation history     │
└─────────────────────────────────────────────────┘
              │
              ▼
        Generate personalized, informed response
```

In [36]:
class UnifiedState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    feedback: str


def unified_investment_assistant(state: UnifiedState, config: RunnableConfig, *, store: BaseStore):
    """An assistant that uses all five memory types."""
    user_id = state["user_id"]
    user_message = state["messages"][-1].content
    
    # 1. PROCEDURAL: Get current instructions
    instructions_item = store.get(("agent", "instructions"), "investment_assistant")
    base_instructions = instructions_item.value["instructions"] if instructions_item else "You are a helpful investment advisory assistant."
    
    # 2. LONG-TERM: Get user profile
    profile_items = list(store.search((user_id, "profile")))
    pref_items = list(store.search((user_id, "preferences")))
    profile_text = "\n".join([f"- {p.key}: {p.value}" for p in profile_items]) if profile_items else "No profile stored."
    
    # 3. SEMANTIC: Search for relevant knowledge
    relevant_knowledge = store.search(("investment", "knowledge"), query=user_message, limit=2)
    knowledge_text = "\n".join([f"- {r.value['text'][:200]}..." for r in relevant_knowledge]) if relevant_knowledge else "No specific knowledge found."
    
    # 4. EPISODIC: Find similar past interactions
    similar_episodes = store.search(("agent", "episodes"), query=user_message, limit=1)
    if similar_episodes:
        ep = similar_episodes[0].value
        episode_text = f"Similar past interaction:\nUser: {ep.get('input', 'N/A')}\nResponse style: {ep.get('feedback', 'N/A')}"
    else:
        episode_text = "No similar past interactions found."
    
    # Build comprehensive system message
    system_message = f"""{base_instructions}

=== USER PROFILE ===
{profile_text}

=== RELEVANT INVESTMENT KNOWLEDGE ===
{knowledge_text}

=== LEARNING FROM EXPERIENCE ===
{episode_text}

Use all of this context to provide the best possible personalized response."""
    
    # 5. SHORT-TERM: Full conversation history is automatically managed by the checkpointer
    # Use summarization for long conversations
    trimmed_messages = summarize_conversation(state["messages"], max_messages=6)
    
    messages = [SystemMessage(content=system_message)] + trimmed_messages
    response = llm.invoke(messages)
    return {"messages": [response]}


def unified_feedback_node(state: UnifiedState, config: RunnableConfig, *, store: BaseStore):
    """Update procedural memory based on feedback."""
    feedback = state.get("feedback", "")
    if not feedback:
        return {}
    
    item = store.get(("agent", "instructions"), "investment_assistant")
    if item is None:
        return {}
    
    current = item.value
    reflection_prompt = f"""Update these instructions based on feedback:

Current: {current['instructions']}
Feedback: {feedback}

Output only the updated instructions."""
    
    response = llm.invoke([HumanMessage(content=reflection_prompt)])
    store.put(
        ("agent", "instructions"),
        "investment_assistant",
        {"instructions": response.content, "version": current["version"] + 1}
    )
    print(f"Procedural memory updated to v{current['version'] + 1}")
    return {}


def unified_route(state: UnifiedState) -> str:
    return "feedback" if state.get("feedback") else "end"


# Build the unified graph
unified_builder = StateGraph(UnifiedState)
unified_builder.add_node("assistant", unified_investment_assistant)
unified_builder.add_node("feedback", unified_feedback_node)

unified_builder.add_edge(START, "assistant")
unified_builder.add_conditional_edges("assistant", unified_route, {"feedback": "feedback", "end": END})
unified_builder.add_edge("feedback", END)

# Compile with both checkpointer (short-term) and store (all other memory types)
unified_graph = unified_builder.compile(
    checkpointer=MemorySaver(),
    store=semantic_store
)

print("Unified investment assistant ready with all 5 memory types!")

Unified investment assistant ready with all 5 memory types!


In [37]:
# Test the unified assistant
config = {"configurable": {"thread_id": "unified_thread_1"}}

# First interaction - should use semantic + long-term + episodic memory
response = unified_graph.invoke(
    {
        "messages": [HumanMessage(content="What investment strategy would you recommend given my profile?")],
        "user_id": "user_alex",  # Alex has moderate risk tolerance and ESG preferences!
        "feedback": ""
    },
    config
)

print("User: What investment strategy would you recommend given my profile?")
print(f"\nAssistant: {response['messages'][-1].content}")
print("\n" + "="*60)
print("Memory types used:")
print("  Long-term: Knows Alex's risk tolerance, portfolio, and ESG preferences")
print("  Semantic: Retrieved investment knowledge from Stone Ridge letter")
print("  Episodic: May use similar advisory episode as reference")
print("  Procedural: Following current instructions")
print("  Short-term: Will remember this in follow-up questions")

User: What investment strategy would you recommend given my profile?

Assistant: To provide a tailored investment strategy, I need to understand more about your specific situation. Here are a few clarifying questions:

1. **Risk Tolerance**: How would you describe your risk tolerance? Are you more conservative, moderate, or aggressive in your investment approach?
   
2. **Investment Goals**: What are your primary investment goals? Are you looking for long-term growth, income generation, capital preservation, or a combination of these?

3. **Time Horizon**: What is your investment time horizon? Are you investing for a short-term goal (1-3 years), medium-term (3-10 years), or long-term (10+ years)?

Once I have this information, I can provide you with actionable insights based on evidence-based investment strategies.

Memory types used:
  Long-term: Knows Alex's risk tolerance, portfolio, and ESG preferences
  Semantic: Retrieved investment knowledge from Stone Ridge letter
  Episodic: M

In [38]:
# Follow-up question (tests short-term memory)
response = unified_graph.invoke(
    {
        "messages": [HumanMessage(content="Can you tell me more about the alternative investments you mentioned?")],
        "user_id": "user_alex",
        "feedback": ""
    },
    config  # Same thread
)

print("User: Can you tell me more about the alternative investments you mentioned?")
print(f"\nAssistant: {response['messages'][-1].content}")
print("\nNotice: The agent remembers the context from the previous message!")

User: Can you tell me more about the alternative investments you mentioned?

Assistant: Certainly! Alternative investments refer to asset classes that fall outside of traditional investments like stocks, bonds, and cash. Here are some key types of alternative investments and their characteristics:

1. **Real Estate**: Investing in physical properties or real estate investment trusts (REITs). Real estate can provide rental income and potential appreciation, but it also comes with risks such as market fluctuations and property management challenges.

2. **Private Equity**: Involves investing in private companies or buyouts of public companies. This can offer high returns but typically requires a longer investment horizon and is less liquid than public equity.

3. **Hedge Funds**: These funds employ various strategies to generate returns, including long/short equity, arbitrage, and global macro. They often require high minimum investments and can have complex fee structures.

4. **Commodi

---
## ❓ Question #3:

How would you decide what constitutes a **"successful" investment advisory interaction** worth storing as an episode? What metadata should you store alongside the episode?

Consider:
- Explicit feedback (thumbs up) vs implicit signals
- User engagement (did they ask follow-up questions?)
- Objective outcomes vs subjective satisfaction
- Privacy implications of storing interaction data

##### Answer:

Some things that would make an interaction more worthwhile to store would be

* If the user engaged with the response. E.g., asked more questions, or gave positive feedback ("thanks," etc.). Those would generally be indications that the interaction was working well.
* If the contents of the interaction indicated a successful response (e.g., "I found this information") versus a negative response (e.g., "I couldn't find anything").
* If the interaction is a common interaction across multiple users, or even the same user in multiple sessions.

Some possible metadata might be

* Boolean or basic enum-type data for indicating the type of episode or some category.
* General sentiment (this was a positive interaction versus a negative one).
* Small summaries of the topic in question.
* In a multi-agent system, it could also store the specialized agent that was used to respond.

## ❓ Question #4:

For a **production investment advisory assistant**, which memory types need persistent storage (PostgreSQL) vs in-memory? How would you handle memory across multiple agent instances (e.g., Market Outlook Agent, Strategy Agent, Risk Management Agent)?

Consider:
- Which memories are user-specific vs shared?
- Consistency requirements across agents
- Memory expiration and cleanup policies
- Namespace strategy for multi-agent systems

##### Answer:

In a production system, long-term, semantic, and episodic memory should be kept in persistent storage, while short-term and procedural can be in-memory.

Across multiple agent instances, semantic and episodic memory should probably be stored along with a key for the agent being used. That way the agent can tailor its responses for its purpose instead of using a generic memory model. Long-term memory could generally be shared across different agents. Although it's possible that someone may want slightly different preferences or behavior when interacting with different agents, in which case those pieces of memory need to be keyed along with the agent.

---
## 🏗️ Activity #2: Investment Memory Dashboard

Build an investment tracking system that:
1. Tracks investment metrics over time (portfolio value, risk score, allocation drift)
2. Uses semantic memory to find relevant investment advice
3. Uses episodic memory to recall what advisory approaches worked before
4. Uses procedural memory to adapt advice style
5. Provides a synthesized "investment summary"

### Requirements:
- Store at least 3 investment metrics per user
- Track metrics over multiple "days" (simulated)
- Agent should reference historical data in responses
- Generate a personalized investment summary

In [40]:
### YOUR CODE HERE ###

# Step 1: Define investment metrics schema and storage functions
from datetime import datetime, timedelta

def log_investment_metric(store, user_id: str, date: str, metric_type: str, value: float, notes: str = ""):
    """Log an investment metric for a user."""
    metrics_namespace = (user_id, "metrics", metric_type)
    
    # Create a unique key using date
    key = f"metric_{date}"
    
    metric_data = {
        "date": date,
        "metric_type": metric_type,
        "value": value,
        "notes": notes,
        "text": f"On {date}, {metric_type} was {value}. {notes}"  # For semantic search
    }
    
    store.put(metrics_namespace, key, metric_data)


def get_investment_history(store, user_id: str, metric_type: str = None, days: int = 7) -> list:
    """Get investment history for a user."""
    if metric_type:
        # Get specific metric type
        metrics_namespace = (user_id, "metrics", metric_type)
        items = list(store.search(metrics_namespace))
    else:
        # Get all metric types - need to search all possible namespaces
        all_metrics = []
        for mtype in ["portfolio_value", "risk_score", "allocation_drift"]:
            metrics_namespace = (user_id, "metrics", mtype)
            items = list(store.search(metrics_namespace))
            all_metrics.extend(items)
        items = all_metrics
    
    # Sort by date (most recent first)
    sorted_items = sorted(items, key=lambda x: x.value.get("date", ""), reverse=True)
    
    # Return up to 'days' worth of data
    return sorted_items[:days] if days else sorted_items


def calculate_metric_trend(history: list) -> str:
    """Calculate if a metric is trending up, down, or stable."""
    if len(history) < 2:
        return "insufficient data"
    
    values = [item.value["value"] for item in history]
    
    # Compare most recent to oldest in the set
    recent_avg = sum(values[:3]) / min(3, len(values))
    older_avg = sum(values[-3:]) / min(3, len(values[-3:]))
    
    change_pct = ((recent_avg - older_avg) / older_avg) * 100 if older_avg != 0 else 0
    
    if change_pct > 2:
        return f"increasing (+{change_pct:.1f}%)"
    elif change_pct < -2:
        return f"decreasing ({change_pct:.1f}%)"
    else:
        return f"stable ({change_pct:.1f}%)"


# Step 2: Create sample investment data for a user (simulate a week)

# Create a dedicated store for the dashboard
dashboard_store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

# Copy existing semantic and episodic memories to the new store
# Copy investment facts
for key, value in investment_facts:
    dashboard_store.put(("investment", "facts"), key, value)

# Copy episodic memories
for i, episode in enumerate(successful_episodes):
    dashboard_store.put(
        ("agent", "episodes"),
        f"episode_{i}",
        {
            "text": episode["situation"],
            **episode
        }
    )

# Copy procedural instructions
dashboard_store.put(
    ("agent", "instructions"),
    "investment_assistant",
    {"instructions": initial_instructions, "version": 1}
)

# Create sample user profile
user_id = "user_jennifer"
profile_namespace = (user_id, "profile")

dashboard_store.put(profile_namespace, "name", {"value": "Jennifer Martinez"})
dashboard_store.put(profile_namespace, "portfolio_size", {"value": "$750K"})
dashboard_store.put(profile_namespace, "risk_tolerance", {"value": "moderate"})
dashboard_store.put(profile_namespace, "investment_horizon", {"value": "15 years"})

print(f"Created profile for {user_id}")

# Simulate a week of investment metrics
base_date = datetime(2026, 2, 16)  # Start a week ago

# Portfolio value - generally increasing with some volatility
portfolio_values = [745000, 748500, 746200, 751800, 749300, 753600, 756400]

# Risk score (0-100, where higher = more risk) - increasing trend
risk_scores = [42, 43, 45, 44, 47, 48, 50]

# Allocation drift (% deviation from target) - increasing trend (concerning)
allocation_drifts = [2.1, 2.5, 3.2, 3.8, 4.5, 5.1, 6.2]

for i in range(7):
    date = (base_date + timedelta(days=i)).strftime("%Y-%m-%d")
    
    # Log portfolio value
    log_investment_metric(
        dashboard_store,
        user_id,
        date,
        "portfolio_value",
        portfolio_values[i],
        f"Market volatility, {'gain' if portfolio_values[i] > (portfolio_values[i-1] if i > 0 else portfolio_values[0]) else 'loss'}"
    )
    
    # Log risk score
    log_investment_metric(
        dashboard_store,
        user_id,
        date,
        "risk_score",
        risk_scores[i],
        "Risk assessment based on portfolio composition"
    )
    
    # Log allocation drift
    log_investment_metric(
        dashboard_store,
        user_id,
        date,
        "allocation_drift",
        allocation_drifts[i],
        f"Drift from target allocation {'within' if allocation_drifts[i] <= 5 else 'exceeds'} threshold"
    )

print(f"Logged 7 days of metrics for {user_id}")
print(f"  - Portfolio values: ${min(portfolio_values):,.0f} to ${max(portfolio_values):,.0f}")
print(f"  - Risk scores: {min(risk_scores)} to {max(risk_scores)}")
print(f"  - Allocation drift: {min(allocation_drifts):.1f}% to {max(allocation_drifts):.1f}%")


# Step 3: Build an investment dashboard agent that:
#   - Retrieves user's investment history
#   - Searches for relevant advice based on patterns
#   - Uses episodic memory for what worked before
#   - Generates a personalized summary

class DashboardState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def investment_dashboard_agent(state: DashboardState, config: RunnableConfig, *, store: BaseStore):
    """Dashboard agent that provides comprehensive investment summaries using all memory types."""
    user_id = state["user_id"]
    user_message = state["messages"][-1].content
    
    # 1. PROCEDURAL: Get current instructions
    instructions_item = store.get(("agent", "instructions"), "investment_assistant")
    base_instructions = instructions_item.value["instructions"] if instructions_item else "You are a helpful investment dashboard assistant."
    
    # 2. LONG-TERM: Get user profile
    profile_items = list(store.search((user_id, "profile")))
    profile_dict = {item.key: item.value["value"] for item in profile_items}
    profile_text = "\n".join([f"- {k}: {v}" for k, v in profile_dict.items()]) if profile_dict else "No profile stored."
    
    # 3. Get historical metrics for all types
    portfolio_history = get_investment_history(store, user_id, "portfolio_value", days=7)
    risk_history = get_investment_history(store, user_id, "risk_score", days=7)
    drift_history = get_investment_history(store, user_id, "allocation_drift", days=7)
    
    # Calculate trends
    portfolio_trend = calculate_metric_trend(portfolio_history)
    risk_trend = calculate_metric_trend(risk_history)
    drift_trend = calculate_metric_trend(drift_history)
    
    # Format historical data
    metrics_summary = f"""
PORTFOLIO VALUE (7 days):
- Current: ${portfolio_history[0].value['value']:,.0f} (Trend: {portfolio_trend})
- Week ago: ${portfolio_history[-1].value['value']:,.0f}
- Change: ${portfolio_history[0].value['value'] - portfolio_history[-1].value['value']:,.0f}

RISK SCORE (7 days):
- Current: {risk_history[0].value['value']}/100 (Trend: {risk_trend})
- Week ago: {risk_history[-1].value['value']}/100
- Change: {risk_history[0].value['value'] - risk_history[-1].value['value']:+.0f} points

ALLOCATION DRIFT (7 days):
- Current: {drift_history[0].value['value']:.1f}% from target (Trend: {drift_trend})
- Week ago: {drift_history[-1].value['value']:.1f}%
- Status: {"EXCEEDS 5% THRESHOLD" if drift_history[0].value['value'] > 5 else "Within acceptable range"}
"""
    
    # 4. SEMANTIC: Search for relevant advice based on the query
    relevant_facts = store.search(("investment", "facts"), query=user_message, limit=2)
    facts_text = "\n".join([f"- {r.value['text']}" for r in relevant_facts]) if relevant_facts else "No specific facts found."
    
    # 5. EPISODIC: Find similar past interactions
    similar_episodes = store.search(("agent", "episodes"), query=user_message, limit=1)
    if similar_episodes:
        ep = similar_episodes[0].value
        episode_text = f"Past interaction style: {ep.get('feedback', 'N/A')}"
    else:
        episode_text = "No similar past interactions."
    
    # Build comprehensive system message
    system_message = f"""{base_instructions}

=== USER PROFILE ===
{profile_text}

=== HISTORICAL METRICS ===
{metrics_summary}

=== RELEVANT INVESTMENT KNOWLEDGE ===
{facts_text}

=== LEARNING FROM EXPERIENCE ===
{episode_text}

You are providing an investment dashboard summary. Use the historical data to identify trends and provide actionable insights. Be data-driven and specific."""
    
    messages = [SystemMessage(content=system_message)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# Build the dashboard graph
dashboard_builder = StateGraph(DashboardState)
dashboard_builder.add_node("dashboard", investment_dashboard_agent)
dashboard_builder.add_edge(START, "dashboard")
dashboard_builder.add_edge("dashboard", END)

dashboard_graph = dashboard_builder.compile(
    checkpointer=MemorySaver(),
    store=dashboard_store
)

print("\n" + "="*80)
print("Investment Memory Dashboard Ready!")
print("="*80)


# Step 4: Test the dashboard
print("\n" + "="*80)
print("TEST 1: Weekly Performance Summary")
print("="*80)

config = {"configurable": {"thread_id": "dashboard_thread_1"}}

response = dashboard_graph.invoke(
    {
        "messages": [HumanMessage(content="Give me a summary of my portfolio performance this week")],
        "user_id": user_id
    },
    config
)

print(f"\nUser: Give me a summary of my portfolio performance this week")
print(f"\nDashboard Response:")
print(response['messages'][-1].content)


print("\n" + "="*80)
print("TEST 2: Volatility Concern")
print("="*80)

config2 = {"configurable": {"thread_id": "dashboard_thread_2"}}

response2 = dashboard_graph.invoke(
    {
        "messages": [HumanMessage(content="My portfolio has been volatile lately. What might help?")],
        "user_id": user_id
    },
    config2
)

print(f"\nUser: My portfolio has been volatile lately. What might help?")
print(f"\nDashboard Response:")
print(response2['messages'][-1].content)


print("\n" + "="*80)
print("TEST 3: Allocation Drift Alert")
print("="*80)

config3 = {"configurable": {"thread_id": "dashboard_thread_3"}}

response3 = dashboard_graph.invoke(
    {
        "messages": [HumanMessage(content="Should I be concerned about my allocation drift?")],
        "user_id": user_id
    },
    config3
)

print(f"\nUser: Should I be concerned about my allocation drift?")
print(f"\nDashboard Response:")
print(response3['messages'][-1].content)


print("\n" + "="*80)
print("MEMORY TYPES DEMONSTRATION:")
print("="*80)
print("✓ SHORT-TERM: Conversation context maintained within each thread")
print("✓ LONG-TERM: User profile (Jennifer, $750K, moderate risk, 15 year horizon)")
print("✓ SEMANTIC: Retrieved relevant investment facts based on query meaning")
print("✓ EPISODIC: Used past successful interaction styles as reference")
print("✓ PROCEDURAL: Followed current dashboard instructions for data-driven advice")
print("\nAll 5 memory types working together to provide personalized investment insights!")

Created profile for user_jennifer
Logged 7 days of metrics for user_jennifer
  - Portfolio values: $745,000 to $756,400
  - Risk scores: 42 to 50
  - Allocation drift: 2.1% to 6.2%

Investment Memory Dashboard Ready!

TEST 1: Weekly Performance Summary

User: Give me a summary of my portfolio performance this week

Dashboard Response:
Here's a summary of your portfolio performance over the past week:

### Portfolio Overview
- **Current Portfolio Value:** $756,400
- **Value Change:** Increased by $11,400 (approximately 1.5%)
- **Trend:** Stable (0.9% increase)

### Risk Assessment
- **Current Risk Score:** 50/100
- **Change in Risk Score:** Increased by 8 points (+11.5%)
- **Trend:** Increasing, indicating a rise in perceived risk within your portfolio.

### Allocation Drift
- **Current Allocation Drift:** 6.2% from target
- **Change in Allocation Drift:** Increased from 2.1% last week (+102.6%)
- **Status:** Exceeds the 5% threshold, suggesting that your portfolio may need rebalancing 

---
## Summary

In this module, we explored the **5 memory types** from the CoALA framework:

| Memory Type | LangGraph Component | Scope | Investment Use Case |
|-------------|---------------------|-------|-------------------|
| **Short-term** | `MemorySaver` + `thread_id` | Within thread | Current consultation |
| **Long-term** | `InMemoryStore` + namespaces | Across threads | User profile, goals, constraints |
| **Semantic** | Store + embeddings + `search()` | Across threads | Investment knowledge retrieval |
| **Episodic** | Store + few-shot examples | Across threads | Past successful interactions |
| **Procedural** | Store + self-reflection | Across threads | Self-improving instructions |

### Key Takeaways:

1. **Memory transforms chatbots into advisors** - Persistence enables personalization
2. **Different memory types serve different purposes** - Choose based on your use case
3. **Context management is critical** - Trim and summarize to stay within limits
4. **Episodic memory enables learning** - Show, don't just tell
5. **Procedural memory enables adaptation** - Agents can improve themselves

### Production Considerations:

- Use `PostgresSaver` instead of `MemorySaver` for persistent checkpoints
- Use `PostgresStore` instead of `InMemoryStore` for persistent long-term memory
- Consider TTL (Time-to-Live) policies for automatic memory cleanup
- Implement proper access controls for user data
- Ensure compliance with financial regulations for investment advisory data

### Further Reading:

- [LangGraph Memory Documentation](https://langchain-ai.github.io/langgraph/concepts/memory/)
- [CoALA Paper](https://arxiv.org/abs/2309.02427) - Cognitive Architectures for Language Agents
- [LangGraph Platform](https://docs.langchain.com/langgraph-platform/) - Managed infrastructure for production